# Section 3. RAG: 검색한 문서만 근거로 답하기

RAG는 모델이 모든 것을 기억한다고 가정하지 않고, 먼저 관련 문서를 찾은 뒤 그 문서만 근거로 답하게 만드는 방식입니다.

이 섹션의 목표:

- 아주 단순한 검색기를 직접 만들어 RAG의 구조를 봅니다.
- 검색 결과와 답변 근거를 분리해서 확인합니다.
- 문서에 없는 질문은 “답할 수 없음”으로 처리합니다.


## 작은 corpus 준비

실제 서비스에서는 PDF, 웹문서, DB, vector DB를 쓰지만, 여기서는 원리를 보기 위해 작은 문서 4개만 사용합니다.


In [ ]:
CORPUS = [
    {
        "source_id": "support_triage",
        "text": "Support triage routes billing requests to the finance queue. Urgent security issues go to the incident channel. Uncertain cases use a fallback path for review.",
    },
    {
        "source_id": "api_onboarding",
        "text": "API onboarding requires authentication, environment variables, awareness of rate limits, and one minimal smoke test before larger experiments.",
    },
    {
        "source_id": "rag_quality",
        "text": "A grounded RAG answer should cite the retrieved source, quote only text that appears in that source, and say not answered when the corpus lacks evidence.",
    },
    {
        "source_id": "release_notes",
        "text": "Release notes summarize user-visible changes, known limitations, and migration steps. They are not a source of private pricing policy.",
    },
]


## LangChain Document로 source_id 유지하기

RAG에서는 문서 본문만 넘기면 출처를 잃기 쉽습니다. LangChain의 `Document`는 `page_content`와 `metadata`를 함께 들고 다니게 해줍니다.


In [ ]:
from langchain_core.documents import Document

DOCUMENTS = [
    Document(page_content=item["text"], metadata={"source_id": item["source_id"]})
    for item in CORPUS
]

for doc in DOCUMENTS:
    print(doc.metadata["source_id"], "=>", doc.page_content[:60] + "...")


## 간단한 lexical retrieval

아래 검색기는 query 단어가 문서에 얼마나 겹치는지로 점수를 매깁니다. 고급 검색기는 아니지만 RAG의 입구를 이해하기에는 충분합니다.


In [ ]:
import re

def tokenize(text: str) -> set[str]:
    return set(re.findall(r"[a-zA-Z0-9_]+", text.lower()))

def retrieve(query: str, top_k: int = 2) -> list[dict]:
    query_tokens = tokenize(query)
    scored = []
    for doc in CORPUS:
        score = len(query_tokens & tokenize(doc["text"]))
        if score:
            scored.append((score, doc))
    scored.sort(key=lambda item: item[0], reverse=True)
    return [doc for _, doc in scored[:top_k]]

retrieve("What is needed for API onboarding?")


## API key 입력

각 실습 노트북은 독립적으로 실행됩니다. 아래 셀의 따옴표 안에 수업용 OpenAI API key를 붙여넣고 실행하세요.

- key가 들어간 노트북은 그대로 공유하지 않습니다.
- 제출하거나 화면 공유하기 전에는 key 문자열을 지웁니다.
- 이 자료에서는 `.env` 파일을 쓰지 않습니다. 학생이 열어야 할 파일 수를 줄이기 위해 노트북 안에서 직접 입력합니다.


In [ ]:
import os

OPENAI_API_KEY = "여기에_수업용_API_KEY를_붙여넣으세요"
OPENAI_MODEL = "gpt-5.4-mini"

if not OPENAI_API_KEY or OPENAI_API_KEY.startswith("여기에_"):
    raise ValueError("OPENAI_API_KEY 자리에 수업용 API key를 붙여넣은 뒤 다시 실행하세요.")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("API key가 현재 노트북 실행 세션에 설정되었습니다.")


## 검색 결과를 근거로 답하기

LLM에게 전체 corpus가 아니라 검색된 문서만 보여줍니다. 답변은 Section 2와 같은 schema로 받습니다.


In [ ]:
from pydantic import BaseModel, Field

class Evidence(BaseModel):
    source_id: str = Field(description="근거 문서 ID")
    quote: str = Field(description="원문에 실제로 포함된 근거 문장")

class GroundedAnswer(BaseModel):
    answered: bool
    answer_ko: str
    evidence: list[Evidence]


In [ ]:
import json
from openai import OpenAI

client = OpenAI()
question = "What is needed before API onboarding experiments?"
retrieved = retrieve(question, top_k=2)
context = "\n\n".join(f"[{d['source_id']}] {d['text']}" for d in retrieved)

prompt = f"""
아래 검색 결과만 근거로 질문에 답하세요.
근거가 부족하면 answered=false, answer_ko="제공된 문서에서 확인할 수 없습니다." 로 답하세요.
반드시 JSON만 출력하세요.

검색 결과:
{context}

질문: {question}
"""

response = client.responses.create(
    model=OPENAI_MODEL,
    input=prompt,
    max_output_tokens=600,
)
raw = response.output_text.strip()
print(raw)
answer = GroundedAnswer.model_validate(json.loads(raw))
answer


## 근거 문장 검증

모델이 적은 quote가 실제 검색 문서에 포함되어 있는지 자동으로 확인합니다.


In [ ]:
source_text = {doc["source_id"]: doc["text"] for doc in CORPUS}
for ev in answer.evidence:
    assert ev.source_id in source_text, f"unknown source_id: {ev.source_id}"
    assert ev.quote in source_text[ev.source_id], f"quote not found: {ev.quote}"
print("evidence quote check passed")


## 문서에 없는 질문

아래 질문은 corpus에 가격 정책이 없기 때문에 답하면 안 됩니다.


In [ ]:
missing_question = "What is the private enterprise pricing policy?"
print(retrieve(missing_question, top_k=2))
print("검색 결과가 없거나 근거가 약하면 답하지 않는 것이 RAG의 중요한 규칙입니다.")
